# Rencana Eksekusi Proyek AI: PSU091

Notebook ini memenuhi requirement berikut:
1. Membangun model Deep Learning menggunakan TensorFlow Functional API atau Model Subclassing.
2. Mengimplementasikan komponen kustom (Custom Layer, Custom Loss, Custom Callback).
3. Menyimpan model dalam format .keras.
4. Menyediakan kode sederhana untuk inference.
5. Menyediakan REST API mandiri menggunakan FastAPI.
6. Menggunakan custom training loop dengan tf.GradientTape.
7. Integrasi Generative AI (Gemini) sebagai fitur sekunder.
8. Integrasi TensorBoard untuk logging metrik.
9. Memastikan performa model: akurasi minimal 85% dan MAE maksimal 0.02.


In [3]:
# Install dependencies (jalankan sekali)
import sys
!{sys.executable} -m pip install pandas numpy tensorflow scikit-learn joblib requests tensorboard --quiet


In [4]:
import os
import csv
import json
import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import joblib
import requests

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow Version: {tf.__version__}")


TensorFlow Version: 2.20.0


In [5]:
# 1. Load Dataset data_final_identik.csv
# Dataset memiliki tanda ';' di akhir baris.
# Beberapa nama makanan mengandung koma, jadi parsing harus aman.

dataset_path = os.path.join('..', 'Dataset', 'data', 'data_final_identik.csv')

def load_dataset(path):
    lines = open(path, encoding='utf-8').read().splitlines()
    cleaned_lines = [line.strip().rstrip(';') for line in lines]

    rows = []
    for row in csv.reader(cleaned_lines):
        if not row:
            continue
        # Pastikan panjang kolom 7: nama_makanan, kalori, protein, lemak, karbo, harga, harga_rp
        if len(row) < 7:
            continue
        if len(row) > 7:
            # Gabungkan kelebihan kolom menjadi nama_makanan
            tail = row[-6:]
            name = ','.join(row[:-6])
            row = [name] + tail
        rows.append(row)

    header = rows[0]
    data = rows[1:]
    df = pd.DataFrame(data, columns=header)

    for col in ['kalori', 'protein', 'lemak', 'karbo', 'harga', 'harga_rp']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

df = load_dataset(dataset_path)

# Gunakan harga_rp sebagai harga utama (harga banyak kosong)
df['harga_final'] = df['harga_rp'].fillna(df['harga'])

# Drop baris dengan nilai penting kosong
df = df.dropna(subset=['kalori', 'protein', 'lemak', 'karbo', 'harga_final'])

print('Jumlah sampel bersih:', len(df))
df.head()


Jumlah sampel bersih: 6273


,nama_makanan,kalori,protein,lemak,karbo,harga,harga_rp,harga_final
0,margarine with yoghurt,88.0,0.058,9.8,0.073,NaN,12500.0,12500.0
1,sunflower seed butter,99.0,2.800,8.8,3.700,NaN,39500.0,39500.0
2,hazelnut oil,120.0,0.000,13.6,0.000,NaN,50500.0,50500.0
3,menhaden fish oil,1966.0,0.000,218.0,0.000,NaN,36000.0,36000.0
4,cod liver fish oil,123.0,0.000,13.6,0.000,NaN,61500.0,61500.0


In [6]:
# 2. Feature Engineering
# Total Nutrisi = (Protein*4) + (Karbo*4) + (Lemak*9)
df['total_nutrisi'] = (df['protein'] * 4) + (df['karbo'] * 4) + (df['lemak'] * 9)
# Skor gizi mentah = Total Nutrisi / (Harga + 1)
df['skor_gizi_mentah'] = df['total_nutrisi'] / (df['harga_final'] + 1)

# 3. Normalisasi
X_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

# Input fitur untuk model
feature_cols = ['harga_final', 'kalori', 'protein', 'lemak', 'karbo']
X = X_scaler.fit_transform(df[feature_cols])
y = y_scaler.fit_transform(df[['skor_gizi_mentah']])

# Simpan scaler untuk inference
joblib.dump(X_scaler, 'nutrition_X_scaler.pkl')
joblib.dump(y_scaler, 'nutrition_y_scaler.pkl')

# 4. Split data
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(1000).batch(batch_size)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(batch_size)

print('Preprocessing selesai')


Preprocessing selesai


In [ ]:
# 5. Custom Layer
@tf.keras.utils.register_keras_serializable(package='Custom', name='NutritionFeatureLayer')
class NutritionFeatureLayer(tf.keras.layers.Layer):
    def __init__(self, units=64, **kwargs):
        super(NutritionFeatureLayer, self).__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='glorot_uniform',
            trainable=True,
            name='w_nutrition'
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True,
            name='b_nutrition'
        )

    def call(self, inputs):
        return tf.nn.relu(tf.matmul(inputs, self.w) + self.b)

    def get_config(self):
        config = super(NutritionFeatureLayer, self).get_config()
        config.update({'units': self.units})
        return config

# 6. Custom Loss
@tf.keras.utils.register_keras_serializable(package='Custom', name='CustomHuberLoss')
class CustomHuberLoss(tf.keras.losses.Loss):
    def __init__(self, delta=1.0, **kwargs):
        super(CustomHuberLoss, self).__init__(**kwargs)
        self.delta = delta

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, dtype=tf.float32)
        y_pred = tf.cast(y_pred, dtype=tf.float32)
        error = y_true - y_pred
        is_small_error = tf.abs(error) <= self.delta
        small_error_loss = tf.square(error) / 2
        large_error_loss = self.delta * (tf.abs(error) - (self.delta / 2))
        return tf.where(is_small_error, small_error_loss, large_error_loss)

    def get_config(self):
        config = super(CustomHuberLoss, self).get_config()
        config.update({'delta': self.delta})
        return config

# 7. Custom Callback
class TargetMetricsCallback(tf.keras.callbacks.Callback):
    def __init__(self, target_mae=0.02, target_acc=0.85):
        super(TargetMetricsCallback, self).__init__()
        self.target_mae = target_mae
        self.target_acc = target_acc

    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            return
        val_mae = logs.get('val_mae')
        val_acc = logs.get('val_accuracy')
        if val_mae is not None and val_acc is not None:
            if val_mae <= self.target_mae and val_acc >= self.target_acc:
                print(f'Target tercapai: MAE={val_mae:.4f}, Accuracy={val_acc:.4f}')
                self.model.stop_training = True

print('Custom components siap')


Custom components siap


In [ ]:
# 8. Membangun Model dengan Functional API
inputs = tf.keras.Input(shape=(len(feature_cols),), name='input_features')
x = NutritionFeatureLayer(units=128)(inputs)
x = tf.keras.layers.Dense(64, activation='relu')(x)
x = tf.keras.layers.Dense(32, activation='relu')(x)
outputs = tf.keras.layers.Dense(1, activation='linear', name='output_skor_gizi')(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name='Nutrition_Optimizer_Model')
model.summary()


Model: "Nutrition_Optimizer_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_features (InputLayer)     │ (None, 5)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ nutrition_feature_layer         │ (None, 128)            │           768 │
│ (NutritionFeatureLayer)         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_skor_gizi (Dense)        │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,137 (43.50 KB)

 Trainable params: 11,137 (43.50 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# 9. Custom Training Loop dengan tf.GradientTape + TensorBoard
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
loss_fn = CustomHuberLoss(delta=1.0)

train_mae_metric = tf.keras.metrics.MeanAbsoluteError()
val_mae_metric = tf.keras.metrics.MeanAbsoluteError()
train_loss_metric = tf.keras.metrics.Mean()

def batch_accuracy(y_true, y_pred, tol=0.02):
    # Cast semua ke float32 dulu agar tidak ada type mismatch
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    error = tf.abs(y_true - y_pred)
    correct = tf.cast(error <= tf.cast(tol, tf.float32), tf.float32)
    return tf.reduce_mean(correct)

train_acc_metric = tf.keras.metrics.Mean()
val_acc_metric = tf.keras.metrics.Mean()

current_time = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
train_log_dir = 'logs/gradient_tape/' + current_time + '/train'
val_log_dir = 'logs/gradient_tape/' + current_time + '/val'
train_summary_writer = tf.summary.create_file_writer(train_log_dir)
val_summary_writer = tf.summary.create_file_writer(val_log_dir)

epochs = 100
target_mae = 0.02
target_acc = 0.85
best_val_mae = 1.0

print('Mulai training')
for epoch in range(epochs):
    # Training
    for x_batch_train, y_batch_train in train_dataset:
        with tf.GradientTape() as tape:
            y_pred = model(x_batch_train, training=True)
            loss_value = loss_fn(y_batch_train, y_pred)
        grads = tape.gradient(loss_value, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))

        train_mae_metric.update_state(y_batch_train, y_pred)
        train_loss_metric.update_state(loss_value)
        train_acc_metric.update_state(batch_accuracy(y_batch_train, y_pred))

    with train_summary_writer.as_default():
        tf.summary.scalar('loss', train_loss_metric.result(), step=epoch)
        tf.summary.scalar('mae', train_mae_metric.result(), step=epoch)
        tf.summary.scalar('accuracy', train_acc_metric.result(), step=epoch)

    # Validation
    for x_batch_val, y_batch_val in val_dataset:
        val_pred = model(x_batch_val, training=False)
        val_mae_metric.update_state(y_batch_val, val_pred)
        val_acc_metric.update_state(batch_accuracy(y_batch_val, val_pred))

    with val_summary_writer.as_default():
        tf.summary.scalar('mae', val_mae_metric.result(), step=epoch)
        tf.summary.scalar('accuracy', val_acc_metric.result(), step=epoch)

    train_mae = float(train_mae_metric.result())
    val_mae = float(val_mae_metric.result())
    val_acc = float(val_acc_metric.result())

    if val_mae < best_val_mae:
        best_val_mae = val_mae

    if epoch % 10 == 0 or epoch == epochs - 1:
        print(f'Epoch {epoch+1:02d} | Train MAE: {train_mae:.4f} | Val MAE: {val_mae:.4f} | Val Acc: {val_acc:.4f}')

    # Early stop sesuai requirement
    if val_mae <= target_mae and val_acc >= target_acc:
        print(f'Target tercapai pada epoch {epoch+1}: MAE={val_mae:.4f}, Acc={val_acc:.4f}')
        break

    train_mae_metric.reset_state()
    val_mae_metric.reset_state()
    train_loss_metric.reset_state()
    train_acc_metric.reset_state()
    val_acc_metric.reset_state()

print('Training selesai')
print(f'Best Val MAE: {best_val_mae:.4f}')


Mulai training
Epoch 01 | Train MAE: 0.0063 | Val MAE: 0.0037 | Val Acc: 0.9883
Target tercapai pada epoch 1: MAE=0.0037, Acc=0.9883
Training selesai
Best Val MAE: 0.0037


In [ ]:
# 10. Simpan model ke format .keras
model.save('nutrition_model.keras')
print('Model tersimpan: nutrition_model.keras')


Model tersimpan: nutrition_model.keras


In [ ]:
# 11. Inference sederhana
def predict_nutrition_score(budget, kalori, protein, lemak, karbo):
    X_scaler_loaded = joblib.load('nutrition_X_scaler.pkl')
    y_scaler_loaded = joblib.load('nutrition_y_scaler.pkl')
    with tf.keras.utils.custom_object_scope({
        'NutritionFeatureLayer': NutritionFeatureLayer,
        'CustomHuberLoss': CustomHuberLoss
    }):
        loaded_model = tf.keras.models.load_model('nutrition_model.keras')

    input_df = pd.DataFrame({
        'harga_final': [budget],
        'kalori': [kalori],
        'protein': [protein],
        'lemak': [lemak],
        'karbo': [karbo]
    })

    input_scaled = X_scaler_loaded.transform(input_df)
    pred_scaled = loaded_model.predict(input_scaled, verbose=0)
    pred_value = y_scaler_loaded.inverse_transform(pred_scaled)[0][0]
    return pred_value

# Contoh inference
score = predict_nutrition_score(budget=15000, kalori=600, protein=25, lemak=15, karbo=70)
print('Skor gizi prediksi:', score)


Skor gizi prediksi: 0.03125273


In [23]:
# 12. Generative AI (Gemini) sebagai fitur sekunder
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', 'YOUR_API_KEY_HERE')

def gemini_suggestion(budget, target_kalori, score):
    if GEMINI_API_KEY == 'YOUR_API_KEY_HERE' or not GEMINI_API_KEY:
        return 'Isi GEMINI_API_KEY untuk menggunakan fitur ini.'

    prompt = (
        f'Seorang pengguna memiliki budget Rp{budget} dan target kalori {target_kalori}. '
        f'Skor efisiensi gizinya diprediksi {score:.2f}. '
        'Berikan 2 kalimat saran singkat tentang makanan lokal Indonesia yang disarankan.'
    )

    url = f'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={GEMINI_API_KEY}'
    payload = {'contents': [{'parts': [{'text': prompt}]}]}
    headers = {'Content-Type': 'application/json'}

    try:
        response = requests.post(url, json=payload, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json()
        return data['candidates'][0]['content']['parts'][0]['text']
    except Exception as e:
        return f'Gagal menghubungi Gemini: {e}'

# Contoh penggunaan (Pastikan 'score' didefinisikan di cell sebelumnya, misal saat inference)
print(gemini_suggestion(15000, 600, score))


Untuk mencapai target kalori 600 dengan budget Rp15.000, pertimbangkan **nasi uduk atau nasi kuning** lengkap dengan telur dan tempe orek, karena padat kalori dan terjangkau. Alternatif lainnya adalah **gado-gado atau pecel** yang kaya sayuran dan protein dari tahu/tempe, dengan saus kacang yang mengenyangkan dan sumber kalori baik.


In [24]:
# 13. REST API mandiri dengan FastAPI (contoh kode)
# Simpan sebagai file app.py jika ingin dijalankan di luar notebook.

fastapi_code = r'''
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import pandas as pd
import tensorflow as tf

app = FastAPI()

class NutritionRequest(BaseModel):
    budget: float
    kalori: float
    protein: float
    lemak: float
    karbo: float

@app.on_event('startup')
def load_assets():
    global model, X_scaler, y_scaler
    X_scaler = joblib.load('nutrition_X_scaler.pkl')
    y_scaler = joblib.load('nutrition_y_scaler.pkl')
    with tf.keras.utils.custom_object_scope({
        'NutritionFeatureLayer': NutritionFeatureLayer,
        'CustomHuberLoss': CustomHuberLoss
    }):
        model = tf.keras.models.load_model('nutrition_model.keras')

@app.get('/health')
def health():
    return {'status': 'ok'}

@app.post('/optimizes')
def optimizes(req: NutritionRequest):
    input_df = pd.DataFrame({
        'harga_final': [req.budget],
        'kalori': [req.kalori],
        'protein': [req.protein],
        'lemak': [req.lemak],
        'karbo': [req.karbo]
    })
    input_scaled = X_scaler.transform(input_df)
    pred_scaled = model.predict(input_scaled, verbose=0)
    pred_value = y_scaler.inverse_transform(pred_scaled)[0][0]
    return {'skor_gizi': float(pred_value)}
'''

print(fastapi_code[:400])



from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import pandas as pd
import tensorflow as tf

app = FastAPI()

class NutritionRequest(BaseModel):
    budget: float
    kalori: float
    protein: float
    lemak: float
    karbo: float

@app.on_event('startup')
def load_assets():
    global model, X_scaler, y_scaler
    X_scaler = joblib.load('nutrition_X_scaler.pkl')
    y


In [ ]:
# 14. TensorBoard Visualization (Colab)
%load_ext tensorboard
%tensorboard --logdir logs/gradient_tape


In [ ]:
# 15. Zip log TensorBoard untuk disimpan di repository
!zip -r logs_tensorboard.zip logs/
